# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
<br>
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Authors (by @id): {[a['@id'] for a in metadata.author]}")
print(f"Record sets (@id): {[r['@id'] for r in getattr(metadata, 'recordSet', [])]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Use record set `@id`s to inspect data. Note: You must reference all entities by their `@id`.

In [ ]:
# Extract record sets and their fields by @id

record_set_ids = [r['@id'] for r in getattr(metadata, 'recordSet', [])]

if not record_set_ids:
    print("No record sets found in the metadata.")
else:
    for rsid in record_set_ids:
        print(f"\nRecordSet @id: {rsid}")
        try:
            recset = dataset.metadata.lookup(rsid)
            field_ids = [f['@id'] for f in getattr(recset, 'field', [])]
            print(f"Fields (@id): {field_ids}")
            col_ids = []
            for f in getattr(recset, 'field', []):
                field_cols = f.get('column')
                if field_cols:
                    if isinstance(field_cols, list):
                        col_ids.extend([c['@id'] for c in field_cols])
                    else:
                        col_ids.append(field_cols['@id'])
            print(f"Columns (@id): {col_ids}")
        except Exception as e:
            print(f"Could not load record set {rsid}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame.

Use the record set and field `@id`s from the overview. All entities and columns in the dataset should be referenced by their `@id`.

In [ ]:
# Collect data from each record set using their @id
record_sets = record_set_ids
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

# Display info about the first available record set
if record_sets:
    first_rs = record_sets[0]
    print(f"Columns in {first_rs}: {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on criteria
- Normalizing numeric fields
- Grouping by key attributes

All references use `@id` and variable-driven access.

In [ ]:
# Choose a record set and field for numeric analysis
# Example: Table 1 field for 'age' (make sure to use @id from the schema)

# The following are placeholder ids and must be replaced by actual @id from the overview above.
record_set_id = record_sets[0] if record_sets else None

# Attempt to infer a suitable numeric field (e.g., 'age') by checking columns
numeric_field_candidates = [col for col in dataframes[record_set_id].columns if 'age' in col.lower()]
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None

# Filtering and normalization
if numeric_field:
    threshold = 10
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Grouping - choose overlap with binary/categorical fields
    group_candidates = [col for col in dataframes[record_set_id].columns if ('hirsutism' in col.lower() or 'phenotype' in col.lower())]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis in the record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
E.g., age distribution, hormone correlations, or phenotype groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and record_set_id:
    plt.figure(figsize=(6,4))
    sns.histplot(dataframes[record_set_id][numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field exists, visualize mean age by group
    if group_field:
        plt.figure(figsize=(6,4))
        grouped = dataframes[record_set_id].groupby(group_field)[numeric_field].mean().reset_index()
        sns.barplot(x=group_field, y=numeric_field, data=grouped)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 clinical dataset using `mlcroissant` by referencing all entities with their `@id`. We reviewed metadata, inspected record sets and their fields, extracted tabular data, filtered and transformed records, and visualized distributions. This workflow provides a reproducible approach for exploratory research and clinical review based on detailed schema descriptors.
